<a href="https://colab.research.google.com/github/anubhavtiwari-cyber/ai-ml-internship-maincrafts/blob/main-maincraft/AI_ML_Task3_Model_Validation_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 AI/ML Task 3 — Model Validation, Overfitting Control & Hyperparameter Tuning
**Maincrafts Technology Internship**  
**Dataset:** California Housing Dataset  
**Focus:** Cross-Validation | Overfitting Detection | GridSearchCV Tuning

## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ All libraries imported successfully!')

## Step 2: Load and Prepare Dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('HousePrice')], axis=1)

X = df.drop('HousePrice', axis=1)
y = df['HousePrice']

print(f'Dataset shape: {df.shape}')
print(f'Features: {list(X.columns)}')
print(f'Target: HousePrice')
df.head()

## Step 3: Feature Scaling (Same as Task-2)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print('✅ StandardScaler applied — mean=0, std=1 for all features')
print(X_scaled_df.describe().loc[['mean','std']].round(3))

## Step 4: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42
)
print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

## Step 5: Detect Overfitting (Train vs Test Performance)
### A large gap between training and test RMSE = Overfitting

In [ ]:
# Unconstrained Decision Tree — classic overfitting example
tree_overfit = DecisionTreeRegressor(random_state=42)  # no max_depth
tree_overfit.fit(X_train, y_train)

train_pred = tree_overfit.predict(X_train)
test_pred  = tree_overfit.predict(X_test)

train_rmse = mean_squared_error(y_train, train_pred)**0.5
test_rmse  = mean_squared_error(y_test,  test_pred)**0.5
train_r2   = r2_score(y_train, train_pred)
test_r2    = r2_score(y_test,  test_pred)

print('🚨 Unconstrained Decision Tree (Overfitting Example):')
print(f'   Train RMSE : {train_rmse:.4f}  |  Test RMSE : {test_rmse:.4f}')
print(f'   Train R2   : {train_r2:.4f}   |  Test R2   : {test_r2:.4f}')
print(f'\n   Gap (Test - Train RMSE): {test_rmse - train_rmse:.4f}')
print('   ⚠️  Large gap confirms OVERFITTING — model memorized training data!')

In [ ]:
# Compare across multiple max_depth values to visualize overfitting
depths = [1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20, None]
train_scores, test_scores = [], []

for d in depths:
    t = DecisionTreeRegressor(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_scores.append(r2_score(y_train, t.predict(X_train)))
    test_scores.append(r2_score(y_test, t.predict(X_test)))

depth_labels = [str(d) if d is not None else 'None' for d in depths]

# Graph 1 — Overfitting curve
plt.figure(figsize=(11, 5))
plt.plot(depth_labels, train_scores, 'o-', color='#2196F3', linewidth=2, markersize=7, label='Train R2')
plt.plot(depth_labels, test_scores,  's--', color='#F44336', linewidth=2, markersize=7, label='Test R2')
plt.fill_between(range(len(depths)),
                 [min(tr, te) for tr, te in zip(train_scores, test_scores)],
                 [max(tr, te) for tr, te in zip(train_scores, test_scores)],
                 alpha=0.1, color='red', label='Overfitting Gap')
plt.xticks(range(len(depths)), depth_labels)
plt.xlabel('max_depth (Decision Tree)', fontsize=12)
plt.ylabel('R2 Score', fontsize=12)
plt.title('Graph 1: Overfitting Detection — Train vs Test R2 across max_depth', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('g1_overfitting_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 1 saved!')

## Step 6: Cross-Validation (Reliable Evaluation)

In [ ]:
models_cv = {
    'Linear Regression':        LinearRegression(),
    'Ridge Regression':         Ridge(alpha=1.0),
    'Decision Tree (depth=5)':  DecisionTreeRegressor(max_depth=5, random_state=42),
    'Decision Tree (no limit)': DecisionTreeRegressor(random_state=42),
}

cv_results = {}
print('5-Fold Cross-Validation Results (RMSE):')
print('-'*55)
for name, model in models_cv.items():
    scores = cross_val_score(model, X_scaled_df, y,
                             scoring='neg_root_mean_squared_error', cv=5)
    cv_rmse = -scores
    cv_results[name] = {
        'CV RMSE Mean': round(cv_rmse.mean(), 4),
        'CV RMSE Std':  round(cv_rmse.std(), 4),
        'scores': cv_rmse
    }
    print(f'{name:30s} → Mean: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}')

print('\n💡 Lower mean + lower std = more reliable model')

In [ ]:
# Graph 2 — CV RMSE comparison with error bars
names = list(cv_results.keys())
means = [cv_results[n]['CV RMSE Mean'] for n in names]
stds  = [cv_results[n]['CV RMSE Std']  for n in names]
colors_bar = ['#2196F3','#FF9800','#4CAF50','#F44336']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, means, yerr=stds, color=colors_bar, edgecolor='white',
              capsize=7, width=0.55, error_kw={'linewidth':2})
for b, m, s in zip(bars, means, stds):
    ax.text(b.get_x()+b.get_width()/2, m+s+0.005, f'{m:.3f}\n±{s:.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_title('Graph 2: 5-Fold Cross-Validation RMSE (with std dev)', fontsize=13, fontweight='bold')
ax.set_ylabel('CV RMSE')
ax.tick_params(axis='x', rotation=12)
plt.tight_layout()
plt.savefig('g2_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 2 saved!')

In [ ]:
# Graph 3 — CV fold-wise scores (boxplot)
cv_data = {name: cv_results[name]['scores'] for name in names}
cv_df = pd.DataFrame(cv_data)

fig, ax = plt.subplots(figsize=(11, 5))
bp = cv_df.boxplot(ax=ax, patch_artist=True,
                   boxprops=dict(linewidth=1.5),
                   medianprops=dict(color='red', linewidth=2))
ax.set_title('Graph 3: CV Fold-wise RMSE Distribution (Boxplot)', fontsize=13, fontweight='bold')
ax.set_ylabel('RMSE per Fold')
ax.tick_params(axis='x', rotation=12)
plt.tight_layout()
plt.savefig('g3_cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 3 saved!')

## Step 7: Hyperparameter Tuning Using GridSearchCV

In [ ]:
# --- GridSearchCV for Decision Tree ---
param_grid_dt = {
    'max_depth':        [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10]
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_dt,
    scoring='neg_root_mean_squared_error',
    cv=5,
    verbose=0,
    n_jobs=-1
)
grid_dt.fit(X_train, y_train)

print('🔧 GridSearchCV — Decision Tree')
print(f'   Best Parameters : {grid_dt.best_params_}')
print(f'   Best CV RMSE    : {-grid_dt.best_score_:.4f}')

In [ ]:
# --- GridSearchCV for Ridge ---
param_grid_ridge = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

grid_ridge = GridSearchCV(
    Ridge(),
    param_grid_ridge,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)
grid_ridge.fit(X_train, y_train)

print('🔧 GridSearchCV — Ridge Regression')
print(f'   Best Parameters : {grid_ridge.best_params_}')
print(f'   Best CV RMSE    : {-grid_ridge.best_score_:.4f}')

In [ ]:
# Graph 4 — GridSearch heatmap (DT: max_depth vs min_samples_split)
cv_results_df = pd.DataFrame(grid_dt.cv_results_)
pivot = cv_results_df.pivot_table(
    index='param_max_depth',
    columns='param_min_samples_split',
    values='mean_test_score'
) * -1

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd_r',
            linewidths=0.5, cbar_kws={'label': 'CV RMSE (lower=better)'})
plt.title('Graph 4: GridSearchCV Heatmap — Decision Tree\n(max_depth vs min_samples_split)', fontsize=12, fontweight='bold')
plt.xlabel('min_samples_split')
plt.ylabel('max_depth')
plt.tight_layout()
plt.savefig('g4_gridsearch_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 4 saved!')

In [ ]:
# Graph 5 — Ridge alpha tuning curve
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_cv_scores = []
for a in alphas:
    sc = cross_val_score(Ridge(alpha=a), X_scaled_df, y,
                         scoring='neg_root_mean_squared_error', cv=5)
    ridge_cv_scores.append(-sc.mean())

plt.figure(figsize=(8, 4))
plt.semilogx(alphas, ridge_cv_scores, 'o-', color='#FF9800', linewidth=2.5, markersize=8)
best_alpha_idx = np.argmin(ridge_cv_scores)
plt.scatter([alphas[best_alpha_idx]], [ridge_cv_scores[best_alpha_idx]],
            color='red', s=150, zorder=5, label=f'Best alpha={alphas[best_alpha_idx]}')
plt.xlabel('Alpha (log scale)', fontsize=12)
plt.ylabel('CV RMSE', fontsize=12)
plt.title('Graph 5: Ridge Regression — Alpha Tuning Curve', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('g5_ridge_alpha_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 5 saved!')

## Step 8: Evaluate Optimized Models

In [ ]:
# Baseline models
lr = LinearRegression().fit(X_train, y_train)
ridge_base = Ridge(alpha=1.0).fit(X_train, y_train)

# Tuned models
best_tree  = grid_dt.best_estimator_
best_ridge = grid_ridge.best_estimator_

all_models = {
    'Linear Regression':    lr,
    'Ridge (alpha=1.0)':    ridge_base,
    'Ridge (Tuned)':        best_ridge,
    'Decision Tree (depth=5)': DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train),
    'Decision Tree (Tuned)': best_tree,
}

final_results = {}
print(f'{"Model":35s} | {"RMSE":8s} | {"MAE":8s} | {"R2":8s}')
print('-'*65)
for name, model in all_models.items():
    p = model.predict(X_test)
    rmse = round(mean_squared_error(y_test, p)**0.5, 4)
    mae  = round(mean_absolute_error(y_test, p), 4)
    r2   = round(r2_score(y_test, p), 4)
    final_results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f'{name:35s} | {rmse:8.4f} | {mae:8.4f} | {r2:8.4f}')

best_final = max(final_results, key=lambda k: final_results[k]['R2'])
print(f'\n🏆 Best Model: {best_final}')

## Step 9: Model Comparison Summary + Graphs

In [ ]:
# Graph 6 — Full model comparison (R2)
mnames = list(final_results.keys())
r2s   = [final_results[m]['R2']   for m in mnames]
rmses = [final_results[m]['RMSE'] for m in mnames]
mc = ['#2196F3','#FF9800','#FFC107','#4CAF50','#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
# R2
bars = axes[0].bar(mnames, r2s, color=mc, edgecolor='white', width=0.55)
for b, v in zip(bars, r2s):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.3f}',
                 ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('R2 Score Comparison', fontweight='bold', fontsize=12)
axes[0].set_ylabel('R2 Score')
axes[0].tick_params(axis='x', rotation=15)
# RMSE
bars2 = axes[1].bar(mnames, rmses, color=mc, edgecolor='white', width=0.55)
for b, v in zip(bars2, rmses):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.002, f'{v:.3f}',
                 ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('RMSE Comparison (Lower = Better)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Graph 6: All Models — Final Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('g6_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 6 saved!')

In [ ]:
# Graph 7 — Before vs After Tuning (DT & Ridge)
categories = ['Ridge\n(alpha=1.0)', 'Ridge\n(Tuned)', 'DT\n(depth=5)', 'DT\n(Tuned)']
r2_vals   = [final_results['Ridge (alpha=1.0)']['R2'],
             final_results['Ridge (Tuned)']['R2'],
             final_results['Decision Tree (depth=5)']['R2'],
             final_results['Decision Tree (Tuned)']['R2']]
rmse_vals = [final_results['Ridge (alpha=1.0)']['RMSE'],
             final_results['Ridge (Tuned)']['RMSE'],
             final_results['Decision Tree (depth=5)']['RMSE'],
             final_results['Decision Tree (Tuned)']['RMSE']]

x = np.arange(2)
w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# R2
axes[0].bar(x-w/2, [r2_vals[0], r2_vals[2]], w, label='Before Tuning', color='#90CAF9', edgecolor='white')
axes[0].bar(x+w/2, [r2_vals[1], r2_vals[3]], w, label='After Tuning',  color='#1565C0', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(['Ridge Regression', 'Decision Tree'])
axes[0].set_title('R2 Score: Before vs After Tuning', fontweight='bold')
axes[0].set_ylabel('R2 Score'); axes[0].legend()
for i, v in enumerate([r2_vals[0], r2_vals[2]]):
    axes[0].text(i-w/2, v+0.002, f'{v:.3f}', ha='center', fontsize=9)
for i, v in enumerate([r2_vals[1], r2_vals[3]]):
    axes[0].text(i+w/2, v+0.002, f'{v:.3f}', ha='center', fontsize=9)

# RMSE
axes[1].bar(x-w/2, [rmse_vals[0], rmse_vals[2]], w, label='Before Tuning', color='#FFCDD2', edgecolor='white')
axes[1].bar(x+w/2, [rmse_vals[1], rmse_vals[3]], w, label='After Tuning',  color='#C62828', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(['Ridge Regression', 'Decision Tree'])
axes[1].set_title('RMSE: Before vs After Tuning', fontweight='bold')
axes[1].set_ylabel('RMSE'); axes[1].legend()
for i, v in enumerate([rmse_vals[0], rmse_vals[2]]):
    axes[1].text(i-w/2, v+0.002, f'{v:.3f}', ha='center', fontsize=9)
for i, v in enumerate([rmse_vals[1], rmse_vals[3]]):
    axes[1].text(i+w/2, v+0.002, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Graph 7: Tuning Impact — Before vs After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('g7_before_after_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 7 saved!')

In [ ]:
# Graph 8 — Learning Curve (Best Model)
best_model_obj = all_models[best_final]
train_sizes, train_sc, val_sc = learning_curve(
    best_model_obj, X_scaled_df, y,
    cv=5, scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)
train_mean = -train_sc.mean(axis=1)
val_mean   = -val_sc.mean(axis=1)
train_std  = train_sc.std(axis=1)
val_std    = val_sc.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', color='#2196F3', lw=2, label='Train RMSE')
plt.plot(train_sizes, val_mean,   's--', color='#F44336', lw=2, label='Validation RMSE')
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='#2196F3')
plt.fill_between(train_sizes, val_mean-val_std,     val_mean+val_std,     alpha=0.15, color='#F44336')
plt.xlabel('Training Set Size', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
plt.title(f'Graph 8: Learning Curve — {best_final}', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('g8_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 8 saved!')

In [ ]:
# Graph 9 — Actual vs Predicted (Tuned Best Model)
y_pred_best = best_model_obj.predict(X_test)
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, alpha=0.3, color='#9C27B0', s=12)
ln = [y_test.min(), y_test.max()]
plt.plot(ln, ln, 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual House Price'); plt.ylabel('Predicted House Price')
plt.title(f'Graph 9: Actual vs Predicted — {best_final}', fontsize=13, fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('g9_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 9 saved!')

In [ ]:
# Graph 10 — Bias-Variance tradeoff (DT depth)
depths2 = list(range(1, 16))
tr2, te2 = [], []
for d in depths2:
    t = DecisionTreeRegressor(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    tr2.append(mean_squared_error(y_train, t.predict(X_train))**0.5)
    te2.append(mean_squared_error(y_test,  t.predict(X_test))**0.5)

best_d = grid_dt.best_params_['max_depth']
plt.figure(figsize=(10, 5))
plt.plot(depths2, tr2, 'o-', color='#2196F3', lw=2, label='Train RMSE')
plt.plot(depths2, te2, 's--', color='#F44336', lw=2, label='Test RMSE')
plt.axvline(x=best_d, color='green', ls=':', lw=2.5, label=f'Best depth={best_d} (GridSearchCV)')
plt.xlabel('max_depth', fontsize=12); plt.ylabel('RMSE', fontsize=12)
plt.title('Graph 10: Bias-Variance Tradeoff — Decision Tree', fontsize=13, fontweight='bold')
plt.legend(fontsize=11); plt.tight_layout()
plt.savefig('g10_bias_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 10 saved!')

In [ ]:
# Graph 11 — Residuals: Before vs After Tuning
dt_before = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train)
dt_after  = best_tree

res_before = y_test.values - dt_before.predict(X_test)
res_after  = y_test.values - dt_after.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(res_before, bins=50, color='#FF9800', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', ls='--', lw=2)
axes[0].set_title('Before Tuning (depth=5)', fontweight='bold')
axes[0].set_xlabel('Residual'); axes[0].set_ylabel('Frequency')
axes[1].hist(res_after, bins=50, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', ls='--', lw=2)
axes[1].set_title(f'After Tuning (best params)', fontweight='bold')
axes[1].set_xlabel('Residual')
plt.suptitle('Graph 11: Residual Distribution — Before vs After Tuning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('g11_residuals_before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 11 saved!')

In [ ]:
# Graph 12 — Summary radar-style grouped bar
task2_models = ['Linear Regression', 'Ridge (alpha=1.0)', 'Decision Tree (depth=5)']
task3_models = ['Ridge (Tuned)', 'Decision Tree (Tuned)']

all_names = task2_models + task3_models
all_r2    = [final_results[m]['R2']   for m in all_names]
all_rmse  = [final_results[m]['RMSE'] for m in all_names]
clrs = ['#90CAF9','#FFCC80','#A5D6A7','#1565C0','#2E7D32']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(all_names, all_r2, color=clrs, edgecolor='white', width=0.6)
axes[0].axvline(2.5, color='gray', ls='--', lw=1.5, label='Task 2 | Task 3')
axes[0].set_title('R2 Score — Task 2 vs Task 3 Models', fontweight='bold')
axes[0].set_ylabel('R2'); axes[0].tick_params(axis='x', rotation=18); axes[0].legend()
for b, v in zip(bars, all_r2):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.003, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

bars2 = axes[1].bar(all_names, all_rmse, color=clrs, edgecolor='white', width=0.6)
axes[1].axvline(2.5, color='gray', ls='--', lw=1.5, label='Task 2 | Task 3')
axes[1].set_title('RMSE — Task 2 vs Task 3 Models', fontweight='bold')
axes[1].set_ylabel('RMSE'); axes[1].tick_params(axis='x', rotation=18); axes[1].legend()
for b, v in zip(bars2, all_rmse):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.002, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Graph 12: Task 2 Baseline vs Task 3 Tuned Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('g12_task2_vs_task3.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 12 saved!')

## Step 10: Final Model Selection Justification

In [ ]:
print('=' * 60)
print('        TASK 3 — FINAL MODEL SELECTION JUSTIFICATION')
print('=' * 60)
print(f'\n🏆 Selected Model : {best_final}')
print(f'   R2 Score       : {final_results[best_final]["R2"]}')
print(f'   RMSE           : {final_results[best_final]["RMSE"]}')
print(f'   MAE            : {final_results[best_final]["MAE"]}')
print()
print('📌 Why this model was selected:')
print('   → Highest R2 — explains maximum variance in house prices')
print('   → Lowest RMSE — predictions closest to actual values')
print('   → Hyperparameters tuned via GridSearchCV (5-fold CV)')
print()
print('🛡  How overfitting was controlled:')
print('   → max_depth limits tree growth (prevents memorization)')
print('   → min_samples_split ensures each split is meaningful')
print('   → Cross-validation selected params on multiple data splits')
print()
print('📊 Why cross-validation results are trusted:')
print('   → 5-fold CV evaluates model on 5 different data subsets')
print('   → Reduces dependence on any single train/test split')
print('   → Low standard deviation = stable, generalizable model')
print('=' * 60)

In [ ]:
# Optional: Save best model
import joblib
joblib.dump(best_model_obj, 'task3_best_model.pkl')
joblib.dump(scaler, 'task3_scaler.pkl')
print(f'✅ Best model saved as task3_best_model.pkl')
print('✅ Scaler saved as task3_scaler.pkl')

---
## ✅ Task 3 Complete!

| Deliverable | Status |
|---|---|
| Jupyter Notebook | ✅ Done |
| Overfitting Detection | ✅ Done |
| Cross-Validation (5-fold) | ✅ Done |
| GridSearchCV Tuning (DT + Ridge) | ✅ Done |
| Model Comparison Table | ✅ Done |
| All 12 Graphs | ✅ Done |
| Final Model Justification | ✅ Done |
| Best Model Saved (joblib) | ✅ Done |

**Maincrafts Technology — AI/ML Internship Task 3**